In [1]:
# Install necessary libraries
!pip install transformers datasets accelerate -U

# Import PyTorch and verify GPU (essential for BERT training)
import torch

# Check for CUDA (NVIDIA GPU)
if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"CUDA is available. Using device: {device}")
else:
    device = torch.device("cpu")
    print("CUDA not available. Using CPU. Training will be slow.")

# Check the GPU in Colab
# !nvidia-smi
# Note: You must ensure your Colab runtime is set to GPU (Runtime -> Change runtime type)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.7/47.7 MB 19.8 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
pylibcudf-cu12 25.6.0 requires pyarrow<20.0.0a0,>=14.0.0; platform_machine == "x86_64", but you have pyarrow 22.0.0 which is incompatible.
cudf-cu12 25.6.0 requires pyarrow<20.0.0a0,>=14.0.0; platform_machine == "x86_64", but you have pyarrow 22.0.0 which is incompatible.
CUDA is available. Using device: cuda


In [2]:
# 2. Dataset Loading and Preprocessing
from datasets import load_dataset
from transformers import AutoTokenizer

# 2.1 Load IMDb Dataset
# The 'datasets' library makes this very easy.
dataset = load_dataset("imdb")

# 2.2 Load BERT Tokenizer
# We use AutoTokenizer to automatically pull the correct tokenizer for bert-base-uncased
model_name = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# 2.3 Define the Tokenization Function
# This function handles tokenizing, truncation, and padding for the entire dataset
def tokenize_function(examples):
    # 'text' is the column containing the movie reviews
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=512)

# 2.4 Apply Tokenization to the dataset
tokenized_datasets = dataset.map(tokenize_function, batched=True)

# 2.5 Prepare Data for PyTorch (Rename columns and format)
tokenized_datasets = tokenized_datasets.remove_columns(["text"])
tokenized_datasets = tokenized_datasets.rename_column("label", "labels")
tokenized_datasets.set_format("torch")

# 2.6 Split into Train/Test Subsets
# We take a small subset for quick demonstration
train_size = 2000 # Use a small number like 2000 for fast testing
test_size = 500  # Use a small number like 500 for fast testing

# Slice the tokenized datasets
train_dataset = tokenized_datasets["train"].shuffle(seed=42).select(range(train_size))
test_dataset = tokenized_datasets["test"].shuffle(seed=42).select(range(test_size))

print(f"Train dataset size: {len(train_dataset)}")
print(f"Test dataset size: {len(test_dataset)}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

plain_text/unsupervised-00000-of-00001.p(…):   0%|          | 0.00/42.0M [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/50000 [00:00<?, ? examples/s]

Train dataset size: 2000
Test dataset size: 500


In [4]:
#🤖 3. Model Fine-Tuning
import numpy as np
from transformers import BertForSequenceClassification, Trainer, TrainingArguments
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

# 3.1 Load BertForSequenceClassification
# num_labels=2 for positive (1) and negative (0) sentiment
model = BertForSequenceClassification.from_pretrained(model_name, num_labels=2)
model.to(device) # Move model to GPU

# 3.2 Define Evaluation Metrics
# The Trainer needs a function to compute metrics
def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)

    # Calculate precision, recall, and F1 score
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='binary')

    # Calculate accuracy
    acc = accuracy_score(labels, preds)

    return {
        'accuracy': acc,
        'f1': f1,
        'precision': precision,
        'recall': recall
    }

# 3.3 Define Training Arguments
# 3.3 Define Training Arguments
training_args = TrainingArguments(
    output_dir="./results",               # output directory
    num_train_epochs=3,                   # total number of training epochs
    per_device_train_batch_size=16,       # batch size per device during training
    per_device_eval_batch_size=16,        # batch size for evaluation
    warmup_steps=500,                     # number of warmup steps for learning rate scheduler
    weight_decay=0.01,                    # strength of weight decay
    logging_dir='./logs',                 # directory for storing logs
    logging_steps=100,

    # *** CHANGE THIS LINE ***
    eval_strategy="epoch",                # NEW: Evaluate at the end of each epoch

    save_strategy="epoch",                # Save model at the end of each epoch
    load_best_model_at_end=True,          # Load the best model found during training
)

# 3.4 Initialize the Trainer
trainer = Trainer(
    model=model,                          # the instantiated 🤗 Transformers model to be trained
    args=training_args,                   # training arguments, defined above
    train_dataset=train_dataset,          # training dataset
    eval_dataset=test_dataset,            # evaluation dataset
    compute_metrics=compute_metrics,      # the function to compute metrics
)

# 3.5 Fine-tune the model (This is the training step)
print("Starting fine-tuning...")
trainer.train()
print("Fine-tuning complete.")

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Starting fine-tuning...


/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)


<IPython.core.display.Javascript object>

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: lakshmimaniram22 (lakshmimaniram22-nxtgenai-services) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,0.623700,0.355300,0.854000,0.854291,0.839216,0.869919
2,0.354800,0.259617,0.908000,0.904959,0.920168,0.890244
3,0.262700,0.489486,0.868000,0.878229,0.804054,0.967480


Fine-tuning complete.


In [5]:
#📊 4. Evaluation and Saving
# 4.1 Calculate final metrics on the test set
print("Final Evaluation on Test Set:")
results = trainer.evaluate()

# Print the final metrics
for key, value in results.items():
    print(f"{key}: {value:.4f}")

# Record these results in your Google Docs documentation.

Final Evaluation on Test Set:


eval_loss: 0.2596
eval_accuracy: 0.9080
eval_f1: 0.9050
eval_precision: 0.9202
eval_recall: 0.8902
eval_runtime: 13.2486
eval_samples_per_second: 37.7400
eval_steps_per_second: 2.4150
epoch: 3.0000


In [6]:
# 4.2 Save the final model and tokenizer
output_dir = "./bert_sentiment_classifier"
model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)
print(f"Model and tokenizer saved to {output_dir}")
# This ensures you can load the trained model later without retraining.

Model and tokenizer saved to ./bert_sentiment_classifier


In [7]:
#Custom testing
from transformers import pipeline

# Load the saved model into a pipeline for easy inference
# We point the pipeline to the directory where we saved the model
sentiment_pipeline = pipeline(
    "sentiment-analysis",
    model=output_dir,
    tokenizer=output_dir,
    device=0 if torch.cuda.is_available() else -1 # Use GPU (device 0) if available
)

# Define custom sentences for testing
custom_sentences = [
    "The movie was absolutely mesmerizing and the acting was top-notch!",  # Positive
    "I would not recommend this product; it broke after only one use.",   # Negative
    "It was neither good nor bad, just utterly forgettable.",             # Neutral/Mixed
    "The presentation of the food was disgusting."                       # Negative
]

# Run predictions
print("\nTesting with custom sentences:")
predictions = sentiment_pipeline(custom_sentences)

# Print and document predictions
for sentence, prediction in zip(custom_sentences, predictions):
    # The 'label' will be 'LABEL_0' (negative) or 'LABEL_1' (positive)
    # The 'score' is the model's confidence

    predicted_sentiment = "Positive" if prediction['label'] == 'LABEL_1' else "Negative"

    print(f"\nSentence: {sentence}")
    print(f"Predicted Sentiment: {predicted_sentiment}")
    print(f"Confidence Score: {prediction['score']:.4f}")

# Save and document these predictions in your Trello card description.

Device set to use cuda:0



Testing with custom sentences:

Sentence: The movie was absolutely mesmerizing and the acting was top-notch!
Predicted Sentiment: Positive
Confidence Score: 0.9677

Sentence: I would not recommend this product; it broke after only one use.
Predicted Sentiment: Negative
Confidence Score: 0.8742

Sentence: It was neither good nor bad, just utterly forgettable.
Predicted Sentiment: Negative
Confidence Score: 0.5203

Sentence: The presentation of the food was disgusting.
Predicted Sentiment: Negative
Confidence Score: 0.8901
